# Assignment 1
**Credits**: Federico Ruggeri, Eleonora Mancini, Paolo Torroni

**Keywords**: Sexism Detection, Multi-class Classification, RNNs, Transformers, Huggingface



# Contact
For any doubt, question, issue or help, you can always contact us at the following email addresses:

Teaching Assistants:

- Federico Ruggeri -> federico.ruggeri6@unibo.it
- Eleonora Mancini -> e.mancini@unibo.it

Professor:
- Paolo Torroni -> p.torroni@unibo.it

# Introduction
You are asked to address the [EXIST 2023 Task 2](https://clef2023.clef-initiative.eu/index.php?page=Pages/labs.html#EXIST) on sexism detection.

## Problem Definition

This task aims to categorize the sexist messages according to the intention of the author in one of the following categories: (i) direct sexist message, (ii) reported sexist message and (iii) judgemental message.

### Examples:

#### DIRECT 
The intention was to write a message that is sexist by itself or incites to be sexist, as in:

''*A woman needs love, to fill the fridge, if a man can give this to her in return for her services (housework, cooking, etc), I don’t see what else she needs.*''

#### REPORTED
The intention is to report and share a sexist situation suffered by a woman or women in first or third person, as in:

''*Today, one of my year 1 class pupils could not believe he’d lost a race against a girl.*''

#### JUDGEMENTAL
The intention was to judge, since the tweet describes sexist situations or behaviours with the aim of condemning them.

''*As usual, the woman was the one quitting her job for the family’s welfare…*''

# [Task 1 - 1.0 points] Corpus

We have preparared a small version of EXIST dataset in our dedicated [Github repository](https://github.com/lt-nlp-lab-unibo/nlp-course-material/tree/main/2025-2026/Assignment%201/data).

Check the `A1/data` folder. It contains 3 `.json` files representing `training`, `validation` and `test` sets.


### Dataset Description
- The dataset contains tweets in both English and Spanish.
- There are labels for multiple tasks, but we are focusing on **Task 2**.
- For Task 2, labels are assigned by six annotators.
- The labels for Task 2 represent whether the tweet is non-sexist ('-') or its sexist intention ('DIRECT', 'REPORTED', 'JUDGEMENTAL').







### Example

```
    "203260": {
        "id_EXIST": "203260",
        "lang": "en",
        "tweet": "ik when mandy says “you look like a whore” i look cute as FUCK",
        "number_annotators": 6,
        "annotators": ["Annotator_473", "Annotator_474", "Annotator_475", "Annotator_476", "Annotator_477", "Annotator_27"],
        "gender_annotators": ["F", "F", "M", "M", "M", "F"],
        "age_annotators": ["18-22", "23-45", "18-22", "23-45", "46+", "46+"],
        "labels_task1": ["YES", "YES", "YES", "NO", "YES", "YES"],
        "labels_task2": ["DIRECT", "DIRECT", "REPORTED", "-", "JUDGEMENTAL", "REPORTED"],
        "labels_task3": [
          ["STEREOTYPING-DOMINANCE"],
          ["OBJECTIFICATION"],
          ["SEXUAL-VIOLENCE"],
          ["-"],
          ["STEREOTYPING-DOMINANCE", "OBJECTIFICATION"],
          ["OBJECTIFICATION"]
        ],
        "split": "TRAIN_EN"
      }
    }
```

### Instructions
1. **Download** the `A1/data` folder.
2. **Load** the three JSON files and encode them as ``pandas.DataFrame``.
3. **Aggregate labels** for Task 2 using majority voting and store them in a new dataframe column called `label`. Items without a clear majority will be removed from the dataset.
4. **Filter the DataFrame** to keep only rows where the `lang` column is `'en'`.
5. **Remove unwanted columns**: Keep only `id_EXIST`, `lang`, `tweet`, and `label`.
6. **Encode the `label` column**: Use the following mapping

```
{
    '-': 0,
    'DIRECT': 1,
    'JUDGEMENTAL': 2,
    'REPORTED': 3
}
```

In [73]:
import os
print("Sei in:", os.getcwd())

Sei in: /Users/giorgioscavello/Documents/GitHub/NLP_project/GIO


In [74]:
import json
import pandas as pd
from collections import Counter

# Load and encode the jsons
train_df = pd.DataFrame(json.load(open('../A1/data/training.json'))).transpose()
test_df = pd.DataFrame(json.load(open('../A1/data/test.json'))).transpose()
val_df = pd.DataFrame(json.load(open('../A1/data/validation.json'))).transpose()
dfs = [train_df, val_df, test_df]

print("Before:", train_df.shape, val_df.shape, test_df.shape)

for i, df in enumerate(dfs):
    # Aggregate labels by majority vote, only keep rows with unique majority
    majority_labels = []
    for labels in df["labels_task2"]:
        c = Counter(labels).most_common(2)
        if len(c)==1:
            majority_labels.append(c[0][0])
        elif c[1][1] < c[0][1]:
            majority_labels.append(c[0][0])
        else:
            majority_labels.append(None)
    df["label"] = majority_labels
    df = df.dropna(subset=["label"]).reset_index(drop=True)

    # Filter rows with language "en"
    df = df[df["lang"]=="en"].reset_index(drop=True)
    df = df[["id_EXIST", "lang", "tweet", "label"]]

    # Encode labels as integers
    label_mapping = {"-":0, "DIRECT":1, "JUDGEMENTAL":2, "REPORTED":3}
    reverse_mapping = {v: k for k, v in label_mapping.items()}  # TODO: utile per debug ma togliere dopo
    df["label"] = df["label"].map(label_mapping)

    dfs[i] = df # NOTE: we have to do this otherwise the changes won't persist


train_df, val_df, test_df = dfs
print("After: ", train_df.shape, val_df.shape, test_df.shape)

Before: (6920, 11) (726, 11) (312, 11)
After:  (2873, 4) (150, 4) (280, 4)


# [Task2 - 0.5 points] Data Cleaning
In the context of tweets, we have noisy and informal data that often includes unnecessary elements like emojis, hashtags, mentions, and URLs. These elements may interfere with the text analysis.



### Instructions
- **Remove emojis** from the tweets.
- **Remove hashtags** (e.g., `#example`).
- **Remove mentions** such as `@user`.
- **Remove URLs** from the tweets.
- **Remove special characters and symbols**.
- **Remove specific quote characters** (e.g., curly quotes).
- **Perform lemmatization** to reduce words to their base form.

In [75]:
# ! pip install --upgrade pip
# ! pip install emoji

In [76]:
import re
import os
import nltk
from nltk import pos_tag, word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import emoji
import gensim
import gensim.downloader as gloader
import numpy as np

In [77]:
# Download required NLTK data
# NOTE: this is mostly from the lab, mayeb it's not the most efficient way
# Download required NLTK data
os.makedirs("nltk_data", exist_ok=True)
nltk.download('omw-1.4', download_dir="./nltk_data")
nltk.download('wordnet', download_dir="./nltk_data")
nltk.download('averaged_perceptron_tagger', download_dir="./nltk_data")
nltk.download('averaged_perceptron_tagger_eng', download_dir="./nltk_data")
nltk.download('punkt_tab', download_dir="./nltk_data")
nltk.data.path.append("./nltk_data")

lemmatizer = WordNetLemmatizer()

def pos2wordnet_tag(treebank_tag: str) -> str:
    match treebank_tag[0]:
        case "J":   return wordnet.ADJ
        case "V":   return wordnet.VERB
        case "N":   return wordnet.NOUN
        case "R":   return wordnet.ADV
        case _:     return wordnet.NOUN

def token_lemma(text: str) -> str:
    """
    Tokenize the text, then lemmatize the tokens and then merge the lemmatized tokens into a text
    """
    tokens = word_tokenize(text)    # usese recommended NLTK tokenizer
    tagged_tokens = pos_tag(tokens)
    lemmatized_tokens = [
        lemmatizer.lemmatize(tok.lower(), pos2wordnet_tag(pos))
        for tok, pos in tagged_tokens
    ]
    return " ".join(lemmatized_tokens)

# NOTE: we replace chars with space to avoid merging words during the process, the tokenizer will take care of extra spaces.
def clean_text(text, patterns):
    
    # 1) Remove emojis
    # NOTE: we asked the tutors and they said that it is meant to remove only visible emojis, not text-based ones. We used the emoji library for completeness.
    # NOTE: the tutor said that we could delete or replace with text, since there are already text emoji we can't handle it could be interesting to replace them with text in order to cover them properly. On the other hand text emoji are rare in the dataset and are for sure used in a different way.
    # NOTE; we don't include the regex for emojis here since we are using the emoji library and it is more efficient and complete.
    text = emoji.replace_emoji(text, replace=' ')    
    
    # Remove other patterns
    for pattern in patterns:
        text = pattern.sub(' ', text) 
    
    # Tokenize, lemmatize and merge
    return token_lemma(text)

[nltk_data] Downloading package omw-1.4 to ./nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to ./nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     ./nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     ./nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to ./nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [78]:
# NOTE: 1 big regex for all patterns would be more efficient, but here we prefer to keep them separate for clarity

patterns = [
    
    # 2) Hashtags
    re.compile(r'#\w+'),
    
    # 3) Mentions
    re.compile(r'@\w+'),
    
    # 4) URLs
    re.compile(r'https?://[^\s]+'), 
    
    # 6) Specific quotes characters, i assume not the regular ones (e.g. " ' )
    # NOTE: we do this here explicitely in order not to implicitly remove them with the special characters pattern
    re.compile(r'[“”‘’«»`´]'), 
    
    # 5) Special characters and symbols
    # NOTE: we asked the tutors as it was ambiguous and they said they meant all non-alphanumeric characters, considering also quotes, parentheses, punctuation, etc.
    re.compile(r'[^a-zA-Z0-9\s]'),

]

for i, df in enumerate(dfs):
    df['cleaned_tweet'] = df["tweet"].apply(lambda text: clean_text(text, patterns))
    
    # TODO: only for train set, maybe apply more complex cleaning:
    # 1) remove noisy chars due to simple removal
    # 2) compression of repeated punctuation
    # 3) MAYBE: html tags, mnore complex and noisy patterns (es. 10%-)
    if i==0: 
        pass
        
    dfs[i] = df    

display(train_df[["tweet", "cleaned_tweet"]])

,tweet,cleaned_tweet
0,FFS! How about laying the blame on the bastard...,ffs how about lay the blame on the bastard who...
1,Writing a uni essay in my local pub with a cof...,write a uni essay in my local pub with a coffe...
2,@UniversalORL it is 2021 not 1921. I dont appr...,it be 2021 not 1921 i dont appreciate that on ...
3,@GMB this is unacceptable. Use her title as yo...,this be unacceptable use her title a you do fo...
4,‘Making yourself a harder target’ basically bo...,make yourself a hard target basically boil dow...
...,...,...
2868,@ShefVaidya Ma'am if I say that you look like ...,ma be if i say that you look like a whore woul...
2869,idk why y’all bitches think having half your a...,idk why y all bitch think have half your as ha...
2870,This has been a part of an experiment with @Wo...,this have be a part of an experiment with what...
2871,"""Take me already"" ""Not yet. You gotta be ready...",take me already not yet you get ta be ready i ...


In [ ]:
def build_word_listing(df):
    vocab = set()
    tokens_vec=[]
    for text in df["cleaned_tweet"]:
        # token_lemma already produces a space-separated string of lemmas
        tokens = word_tokenize(text)
        vocab.update(tokens)
        tokens_vec.append(tokens)
    return sorted(vocab), tokens_vec

word_listing, train_corpus = build_word_listing(train_df)
word_listing_val, train_corpus_val = build_word_listing(val_df)
word_listing_test, train_corpus_test = build_word_listing(test_df)

# [Task 3 - 0.5 points] Text Encoding
To train a neural sexism classifier, you first need to encode text into numerical format.




### Instructions

* Embed words using **GloVe embeddings**.
* You are **free** to pick any embedding dimension.





In [93]:
def load_embedding_model(model_type: str,
                         embedding_dimension: int = 50) -> gensim.models.keyedvectors.KeyedVectors:
    """
    Loads a pre-trained word embedding model via gensim library.

    :param model_type: name of the word embedding model to load.
    :param embedding_dimension: size of the embedding space to consider

    :return
        - pre-trained word embedding model (gensim KeyedVectors object)
    """
    download_path = ""
    if model_type.strip().lower() == 'word2vec':
        download_path = "word2vec-google-news-300"

    elif model_type.strip().lower() == 'glove':
        download_path = "glove-wiki-gigaword-{}".format(embedding_dimension)
    elif model_type.strip().lower() == 'fasttext':
        download_path = "fasttext-wiki-news-subwords-300"
    else:
        raise AttributeError("Unsupported embedding model type! Available ones: word2vec, glove, fasttext")
        
    try:
        emb_model = gloader.load(download_path)
    except ValueError as e:
        print("Invalid embedding model name! Check the embedding dimension:")
        print("Word2Vec: 300")
        print("Glove: 50, 100, 200, 300")
        print('FastText: 300')
        raise e

    return emb_model

In [94]:
embedding_dimension = 50

embedding_model = load_embedding_model(model_type="glove",
                                       embedding_dimension=embedding_dimension)

### What about OOV tokens?
   * All the tokens in the **training** set that are not in GloVe **must** be added to the vocabulary.
   * For the remaining tokens (i.e., OOV in the validation and test sets), you have to assign them a **special token** (e.g., ``<UNK>``) and a **static** embedding.
   * You are **free** to define the static embedding using any strategy (e.g., random, neighbourhood, etc...)



### More about OOV

For a given token:

* **If in train set**: add to vocabulary and assign an embedding (use GloVe if token in GloVe, custom embedding otherwise).
* **If in val/test set**: assign special token if not in vocabulary and assign custom embedding.

Your vocabulary **should**:

* Contain all tokens in train set; or
* Union of tokens in train set and in GloVe $\rightarrow$ we make use of existing knowledge!

In [95]:
def check_OOV_terms(embedding_model: gensim.models.keyedvectors.KeyedVectors,
                    word_listing: list[str]):
    """
    Checks differences between pre-trained embedding model vocabulary
    and dataset specific vocabulary in order to highlight out-of-vocabulary terms.

    :param embedding_model: pre-trained word embedding model (gensim wrapper)
    :param word_listing: dataset specific vocabulary (list)

    :return
        - list of OOV terms
    """
    embedding_vocabulary = set(embedding_model.key_to_index.keys())
    oov = set(word_listing).difference(embedding_vocabulary)
    return list(oov)

In [97]:
oov_terms = check_OOV_terms(embedding_model, word_listing)
oov_percentage = float(len(oov_terms)) * 100 / len(word_listing)
print(f"Total OOV terms: {len(oov_terms)} ({oov_percentage:.2f}%)")
oov_terms_val = check_OOV_terms(embedding_model, word_listing_val)
oov_percentage_val = float(len(oov_terms)) * 100 / len(word_listing_val)
print(f"Total OOV terms in validation: {len(oov_terms_val)} ({oov_percentage_val:.2f}%)")
oov_terms_test = check_OOV_terms(embedding_model, word_listing_test)
oov_percentage_test = float(len(oov_terms)) * 100 / len(word_listing_test)
print(f"Total OOV terms in test: {len(oov_terms_test)} ({oov_percentage_test:.2f}%)")
print("Some OOV terms:", oov_terms[:35])

Total OOV terms: 930 (10.20%)
Total OOV terms in validation: 80 (63.87%)
Total OOV terms in test: 113 (43.85%)
Some OOV terms: ['movt', 'doucheraj', 'trumpanzee', 'alresdy', 'cashmeets', 'unvaxxed', 'forcely', 'akdhajhdajjahshh', 'votewhy', 'ngang', 'othersthrough', 'youyou', 'yr7', 'chuuyas', 'yessssss', 'wildturtle', 'queercoded', 'gqp', '000s', 'fastforward', 'librel', 'parellels', 'bilocal', 'wachana', 'oget', 'homosapiens', 'liiiife', 'referreing', '1yrs', 'ckwits', 'pasiko', 'whop', 'downvoted', 'smartnews', 'ambitionz']


As the 1 term out of 10 on average is out of the vocabulary then the A La Carte approximation was found to create embeddings for these tokens depending on the context.

In [98]:
import numpy as np
import gensim.downloader as gloader
from typing import List, Dict


# Add these functions for A La Carte
def build_alacarte_transform(train_corpus: List[List[str]], 
                             embedding_model,
                             window_size: int = 5):
    """
    Build transformation matrix from training corpus.
    
    :param train_corpus: List of tokenized sentences from training set
    :param embedding_model: Loaded GloVe model
    :param window_size: Context window size
    :return: Transformation matrix A
    """
    context_vectors = {}
    context_counts = {}
    
    # Build context vectors for in-vocabulary words
    for sentence in train_corpus:
        for i, word in enumerate(sentence):
            if word not in embedding_model:
                continue
            
            # Extract context window
            start = max(0, i - window_size)
            end = min(len(sentence), i + window_size + 1)
            context_words = sentence[start:i] + sentence[i+1:end]
            
            # Average context embeddings
            context_vecs = [embedding_model[w] for w in context_words if w in embedding_model]
            
            if len(context_vecs) > 0:
                avg_context = np.mean(context_vecs, axis=0)
                
                if word not in context_vectors:
                    context_vectors[word] = avg_context
                    context_counts[word] = 1
                else:
                    context_vectors[word] += avg_context
                    context_counts[word] += 1
    
    # Average over all occurrences
    for word in context_vectors:
        context_vectors[word] /= context_counts[word]
    
    # Learn transformation matrix via linear regression
    U = np.array([context_vectors[w] for w in context_vectors])
    V = np.array([embedding_model[w] for w in context_vectors])
    
    A = np.linalg.lstsq(U, V, rcond=None)[0].T
    
    return A

A = build_alacarte_transform(train_corpus, embedding_model, window_size=5)

Build the vocabulary

In [99]:
def build_vocabulary_train_union_glove(train_corpus: list, embedding_model) -> set:
    """Build vocabulary as union of training tokens and GloVe vocabulary."""
    train_tokens = set()
    for sentence in train_corpus:
        train_tokens.update(sentence)
    
    glove_vocab = set(embedding_model.index_to_key)
    vocabulary = train_tokens | glove_vocab
    
    print(f"\nVocabulary Statistics:")
    print(f"  - Training tokens: {len(train_tokens)}")
    print(f"  - GloVe vocabulary: {len(glove_vocab)}")
    print(f"  - Union vocabulary: {len(vocabulary)}")
    print(f"  - Train OOV (not in GloVe): {len(train_tokens - glove_vocab)}")
    
    return vocabulary, train_tokens

vocabulary, train_tokens = build_vocabulary_train_union_glove(train_corpus, embedding_model)


Vocabulary Statistics:
  - Training tokens: 9116
  - GloVe vocabulary: 400000
  - Union vocabulary: 400930
  - Train OOV (not in GloVe): 930


In [100]:
from collections import defaultdict
from tensorflow.keras.preprocessing.text import Tokenizer

def create_custom_embeddings_for_train_oov(train_corpus: list,
                                           embedding_model,
                                           transform_matrix: np.ndarray,
                                           embedding_dim: int) -> dict:
    """
    Create custom embeddings using A La Carte for tokens in training but not in GloVe.
    
    :param train_corpus: Training corpus
    :param embedding_model: GloVe model
    :param transform_matrix: A La Carte matrix
    :param embedding_dim: Embedding dimension
    :return: Dictionary of custom embeddings for train OOV tokens
    """
    # Find tokens in training but not in GloVe
    train_tokens = set()
    for sentence in train_corpus:
        train_tokens.update(sentence)
    
    train_oov = train_tokens - set(embedding_model.index_to_key)
    
    print(f"\nCreating custom embeddings for {len(train_oov)} training tokens not in GloVe...")
    
    # Collect contexts for each OOV token in training
    oov_contexts = defaultdict(list)
    
    for sentence in train_corpus:
        for word in sentence:
            if word in train_oov:
                # Get context words (excluding the OOV word, only use words in GloVe)
                context = [w for w in sentence if w != word and w in embedding_model]
                if len(context) > 0:
                    oov_contexts[word].append(context)
    
    # Generate embeddings using A La Carte
    custom_embeddings = {}
    
    for oov_word in train_oov:
        if oov_word in oov_contexts and len(oov_contexts[oov_word]) > 0:
            # Average across all contexts
            all_context_vecs = []
            for context_words in oov_contexts[oov_word]:
                context_embs = [embedding_model[w] for w in context_words]
                avg_context = np.mean(context_embs, axis=0)
                all_context_vecs.append(avg_context)
            
            final_context = np.mean(all_context_vecs, axis=0)
            custom_embeddings[oov_word] = transform_matrix @ final_context
        else:
            # No valid context found - use random initialization
            custom_embeddings[oov_word] = np.random.normal(0, 0.1, embedding_dim)
    
    print(f"  - With A La Carte: {sum(1 for w in train_oov if w in oov_contexts)}")
    print(f"  - Random init: {len(train_oov) - sum(1 for w in train_oov if w in oov_contexts)}")
    
    return custom_embeddings


train_custom_embeddings = create_custom_embeddings_for_train_oov(
    train_corpus,
    embedding_model,
    A,
    embedding_dimension
)


Creating custom embeddings for 930 training tokens not in GloVe...
  - With A La Carte: 930
  - Random init: 0


In [101]:
def build_tokenizer_with_vocabulary(vocabulary: set):
    """
    Build tokenizer with predefined vocabulary (Train ∪ GloVe).
    
    :param vocabulary: Set of all vocabulary words
    :return: Tokenizer object
    """
    tokenizer = Tokenizer(oov_token='<OOV>')  # Special token for val/test OOV
    
    # Create fake documents to force vocabulary
    fake_docs = [' '.join(vocabulary)]
    tokenizer.fit_on_texts(fake_docs)
    
    return tokenizer


tokenizer = build_tokenizer_with_vocabulary(vocabulary)
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding

print(f"\nTokenizer vocabulary size: {vocab_size}")
print(f"OOV token index: {tokenizer.word_index.get('<OOV>', 'Not found')}")


Tokenizer vocabulary size: 340183
OOV token index: 1


In [91]:
test_words=unmatched_words

print("Checking GloVe:")
for word in test_words:
    try:
        vec = embedding_model[word]
        print(f"  {word}: IN GloVe ✓")
    except:
        print(f"  {word}: NOT in GloVe ✗")

# Check: Are these words in training?
train_all_words = set()
for sentence in train_corpus:
    train_all_words.update(sentence)

print("\nChecking training:")
for word in test_words:
    if word in train_all_words:
        print(f"  {word}: IN training ✓")
    else:
        print(f"  {word}: NOT in training ✗")


Checking GloVe:
  coxnews: NOT in GloVe ✗
  latimes: NOT in GloVe ✗
  083: NOT in GloVe ✗
  pbpost: NOT in GloVe ✗
  042: NOT in GloVe ✗
  085: NOT in GloVe ✗
  065: NOT in GloVe ✗
  067: NOT in GloVe ✗
  058: NOT in GloVe ✗
  hearstdc: NOT in GloVe ✗
  082: NOT in GloVe ✗
  eur2004: NOT in GloVe ✗
  066: NOT in GloVe ✗
  073: NOT in GloVe ✗
  057: NOT in GloVe ✗
  062: NOT in GloVe ✗
  038: NOT in GloVe ✗
  096: NOT in GloVe ✗
  timesunion: NOT in GloVe ✗
  068: NOT in GloVe ✗
  64033: NOT in GloVe ✗
  064: NOT in GloVe ✗
  tampabay: NOT in GloVe ✗
  usdoj: NOT in GloVe ✗
  latimescolumnists: NOT in GloVe ✗
  036: NOT in GloVe ✗
  cvw: NOT in GloVe ✗
  069: NOT in GloVe ✗
  acquiremedia: NOT in GloVe ✗
  097: NOT in GloVe ✗
  078: NOT in GloVe ✗
  4425: NOT in GloVe ✗
  3622: NOT in GloVe ✗
  sonderburg: NOT in GloVe ✗
  orthoplex: NOT in GloVe ✗
  5625: NOT in GloVe ✗
  baltsun: NOT in GloVe ✗
  prodmail: NOT in GloVe ✗
  079: NOT in GloVe ✗
  hiberno: NOT in GloVe ✗
  3335: NOT in G

In [102]:
def build_embedding_matrix(tokenizer,
                          embedding_model,
                          train_custom_embeddings: dict,
                          embedding_dim: int) -> np.ndarray:
    """
    Build embedding matrix:
    - GloVe embeddings for words in GloVe
    - Custom embeddings for training tokens not in GloVe
    - Special embedding for <OOV> token (for val/test OOV)
    """
    vocab_size = len(tokenizer.word_index) + 1
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    
    glove_count = 0
    custom_count = 0
    oov_token_count = 0
    padding_count = 1  # Index 0 is always padding
    other_count = 0
    
    for word, idx in tokenizer.word_index.items():
        if word == '<OOV>':
            # Special OOV token - will be updated with A La Carte for val/test
            embedding_matrix[idx] = np.random.normal(0, 0.1, embedding_dim)
            oov_token_count += 1
        elif word in embedding_model:
            # Use GloVe embedding
            embedding_matrix[idx] = embedding_model[word]
            glove_count += 1
        elif word in train_custom_embeddings:
            # Use custom embedding (A La Carte from training)
            embedding_matrix[idx] = train_custom_embeddings[word]
            custom_count += 1
        else:
            # Edge case: word in vocabulary but not in GloVe or custom embeddings
            # This can happen with the union vocabulary
            embedding_matrix[idx] = np.random.normal(0, 0.1, embedding_dim)
            other_count += 1
    
    total_assigned = padding_count + glove_count + custom_count + oov_token_count + other_count
    
    print(f"\nEmbedding Matrix Composition:")
    print(f"  - Padding (index 0): {padding_count}")
    print(f"  - GloVe embeddings: {glove_count}")
    print(f"  - Custom (train OOV via A La Carte): {custom_count}")
    print(f"  - OOV token: {oov_token_count}")
    if other_count > 0:
        print(f"  - Other/Unmatched: {other_count}")
    print(f"  ─────────────────────────────────────")
    print(f"  Total vocabulary size: {vocab_size}")
    print(f"  Sum of assignments: {total_assigned}")
    
    if total_assigned != vocab_size:
        print(f"  ⚠️  Mismatch: {vocab_size - total_assigned} embeddings unaccounted for")
    
    return embedding_matrix

embedding_matrix=build_embedding_matrix(tokenizer, embedding_model, train_custom_embeddings, embedding_dimension)

# Debug: Find unmatched words
unmatched_words = []
for word, idx in tokenizer.word_index.items():
    if (word != '<OOV>' and 
        word not in embedding_model and 
        word not in train_custom_embeddings):
        unmatched_words.append(word)

if unmatched_words:
    print(f"\n⚠️  Found {len(unmatched_words)} unmatched words:")
    print(f"Examples: {unmatched_words[:20]}")



Embedding Matrix Composition:
  - Padding (index 0): 1
  - GloVe embeddings: 337393
  - Custom (train OOV via A La Carte): 930
  - OOV token: 1
  - Other/Unmatched: 1858
  ─────────────────────────────────────
  Total vocabulary size: 340183
  Sum of assignments: 340183

⚠️  Found 1858 unmatched words:
Examples: ['coxnews', 'latimes', '083', 'pbpost', '042', '085', '065', '067', '058', 'hearstdc', '082', 'eur2004', '066', '073', '057', '062', '038', '096', 'timesunion', '068']


In [70]:
def create_valtest_oov_embeddings(df: pd.DataFrame,
                                  tokenizer,
                                  embedding_model,
                                  transform_matrix: np.ndarray,
                                  embedding_dim: int) -> dict:
    """
    Create embeddings for OOV tokens in val/test using A La Carte.
    
    :param df: Validation or test dataframe
    :param tokenizer: Fitted tokenizer
    :param embedding_model: GloVe model
    :param transform_matrix: A La Carte matrix
    :param embedding_dim: Embedding dimension
    :return: Dictionary mapping OOV words to embeddings
    """
    _, corpus = build_word_listing(df)
    
    # Find tokens not in vocabulary
    valtest_tokens = set()
    for sentence in corpus:
        valtest_tokens.update(sentence)
    
    vocab_tokens = set(tokenizer.word_index.keys()) - {'<OOV>'}
    oov_tokens = valtest_tokens - vocab_tokens
    
    print(f"\nFound {len(oov_tokens)} OOV tokens in dataset")
    
    if len(oov_tokens) == 0:
        return {}
    
    # Collect contexts for OOV tokens
    oov_contexts = defaultdict(list)
    
    for sentence in corpus:
        for word in sentence:
            if word in oov_tokens:
                context = [w for w in sentence if w != word and w in embedding_model]
                if len(context) > 0:
                    oov_contexts[word].append(context)
    
    # Generate embeddings
    oov_embeddings = {}
    
    for oov_word in oov_tokens:
        if oov_word in oov_contexts and len(oov_contexts[oov_word]) > 0:
            all_context_vecs = []
            for context_words in oov_contexts[oov_word]:
                context_embs = [embedding_model[w] for w in context_words]
                avg_context = np.mean(context_embs, axis=0)
                all_context_vecs.append(avg_context)
            
            final_context = np.mean(all_context_vecs, axis=0)
            oov_embeddings[oov_word] = transform_matrix @ final_context
        else:
            oov_embeddings[oov_word] = np.random.normal(0, 0.1, embedding_dim)
    
    return oov_embeddings


# Create OOV embeddings for val/test
val_oov_embeddings = create_valtest_oov_embeddings(
    val_df, tokenizer, embedding_model, A, embedding_dimension
)

test_oov_embeddings = create_valtest_oov_embeddings(
    test_df, tokenizer, embedding_model, A, embedding_dimension
)

# Combine val/test OOV embeddings
valtest_oov_embeddings = {**val_oov_embeddings, **test_oov_embeddings}

# Update <OOV> token embedding as average of all val/test OOV embeddings
if len(valtest_oov_embeddings) > 0:
    oov_token_idx = tokenizer.word_index['<OOV>']
    embedding_matrix[oov_token_idx] = np.mean(
        list(valtest_oov_embeddings.values()), axis=0
    )
    print(f"\nUpdated <OOV> token embedding with average of {len(valtest_oov_embeddings)} val/test OOV words")


Found 68 OOV tokens in dataset

Found 96 OOV tokens in dataset

Updated <OOV> token embedding with average of 164 val/test OOV words


# [Task 4 - 1.0 points] Model definition

You are now tasked to define your sexism classifier.




### Instructions

* **Baseline**: implement a Bidirectional LSTM with a Dense layer on top.

* **Stacked**: add an additional Bidirectional LSTM layer to the Baseline model.

**Note**: You are **free** to experiment with hyper-parameters.

### Token to embedding mapping

You can follow two approaches for encoding tokens in your classifier.

### Work directly with embeddings

- Compute the embedding of each input token
- Feed the mini-batches of shape ``(batch_size, # tokens, embedding_dim)`` to your model

### Work with Embedding layer

- Encode input tokens to token ids
- Define a Embedding layer as the first layer of your model
- Compute the embedding matrix of all known tokens (i.e., tokens in your vocabulary)
- Initialize the Embedding layer with the computed embedding matrix
- You are **free** to set the Embedding layer trainable or not

In [12]:
embedding = tf.keras.layers.Embedding(input_dim=vocab_size,
                                      output_dim=embedding_dimension,
                                      weights=[embedding_matrix],
                                      mask_zero=True,                   # automatically masks padding tokens
                                      name='encoder_embedding')

NameError: name 'tf' is not defined

# [Task 5 - 1.0 points] Training and Evaluation

You are now tasked to train and evaluate the Baseline and Stacked models.



### Instructions

* Pick **at least** three seeds for robust estimation.
* Train **all** models on the train set.
* Evaluate **all** models on the validation and test sets.
* Compute macro F1-score, precision, and recall metrics on the validation set.
* Report average and standard deviation measures over seeds for each metric.
* Pick the **best** performing model according to the observed validation set performance (use macro F1-score).

# [Task 6 - 1.0 points] Transformers

In this section, you will use a transformer model specifically trained for hate speech detection, namely [Twitter-roBERTa-base for Hate Speech Detection](https://huggingface.co/cardiffnlp/twitter-roberta-base-hate).




### Relevant Material
- Tutorial 3

### Instructions
- **Load the Tokenizer and Model**

- **Preprocess the Dataset**:
   You will need to preprocess your dataset to prepare it for input into the model. Tokenize your text data using the appropriate tokenizer and ensure it is formatted correctly.

- **Train the Model**:
   Use the `Trainer` to train the model on your training data.

- **Evaluate the Model on the Test Set** using the same metrics used for LSTM-based models.

# [Task 7 - 0.5 points] Error Analysis

After evaluating the model, perform a brief error analysis on the **test set**:

### Instructions

 - Review the results and identify common errors.

 - Summarize your findings regarding the errors and their impact on performance (e.g. but not limited to Out-of-Vocabulary (OOV) words, data imbalance, and performance differences between the custom model and the transformer...)
 - Suggest possible solutions to address the identified errors.

# [Task 8 - 0.5 points] Report

Wrap up your experiment in a short report (up to 2 pages).

### Instructions

* Use the NLP course report template.
* Summarize each task in the report following the provided template.

### Recommendations

The report is **not a copy-paste** of graphs, tables, and command outputs.

* Summarize classification performance in Table format.
* **Do not** report command outputs or screenshots.
* Report learning curves in Figure format.
* The error analysis section should summarize your findings.


# Submission

* **Submit** your report in PDF format.
* **Submit** your python notebook.
* Make sure your notebook is **well organized**, with no temporary code, commented sections, tests, etc...
* You can upload **model weights** in a cloud repository and report the link in the report.

## Bonus Points
Bonus points are arbitrarily assigned based on significant contributions such as:
- Outstanding error analysis
- Masterclass code organization
- Suitable extensions

**Note**: bonus points are only assigned if all task points are attributed (i.e., 6/6).

**Possible Suggestions for Bonus Points:**
- **Try other preprocessing strategies**: e.g., but not limited to, explore techniques tailored specifically for tweets or  methods that are common in social media text.
- **Experiment with other custom architectures or models from HuggingFace**
- **Explore Spanish tweets**: e.g., but not limited to, leverage multilingual models to process Spanish tweets and assess their performance compared to monolingual models.

# FAQ

Please check this frequently asked questions before contacting us

### Trainable Embeddings

You are **free** to define a trainable or non-trainable Embedding layer to load the GloVe embeddings.

### Model architecture

You **should not** change the architecture of a model (i.e., its layers).

However, you are **free** to play with their hyper-parameters.


### Neural Libraries

You are **free** to use any library of your choice to implement the networks (e.g., Keras, Tensorflow, PyTorch, JAX, etc...)

### Robust Evaluation

Each model is trained with at least 3 random seeds.

Task 5 requires you to compute the average performance over the 3 seeds and its corresponding standard deviation.

### Expected Results

Task 2 leaderboard reports around 40-50 F1-score.
However, note that they perform a hierarchical classification.

That said, results around 30-40 F1-score are **expected** given the task's complexity.

### Model Selection for Analysis

To carry out the error analysis you are **free** to either

* Pick examples or perform comparisons with an individual seed run model (e.g., Baseline seed 1337)
* Perform ensembling via, for instance, majority voting to obtain a single model.

### Error Analysis

Some topics for discussion include:
   * Precision/Recall curves.
   * Confusion matrices.
   * Specific misclassified samples.


# The End

Feel free to reach out for questions/doubts!